In [2]:
!pip install scikit-learn gensim pyLDAvis


  Using cached gensim-4.4.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (8.4 kB)
  Using cached pyLDAvis-3.4.1-py3-none-any.whl.metadata (4.2 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 69.0 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import pandas as pd

folder = "/content/drive/MyDrive/Mini"

df_wel  = pd.read_csv(f"{folder}/welfake_clean.csv")
df_net  = pd.read_csv(f"{folder}/fakenewsnet_clean.csv")
df_pred = pd.read_csv(f"{folder}/news_clean.csv")


In [5]:
# The datasets already contain a cleaned column named "text"
wel_texts  = df_wel['text'].astype(str).tolist()
net_texts  = df_net['text'].astype(str).tolist()
pred_texts = df_pred['text'].astype(str).tolist()


In [6]:

def tokenize(texts):
    return [
        [word for word in t.split()]
        for t in texts
    ]

wel_tokens  = tokenize(wel_texts)
net_tokens  = tokenize(net_texts)
pred_tokens = tokenize(pred_texts)


In [7]:
from gensim.corpora import Dictionary

def build_dict(tokens):
    dictionary = Dictionary(tokens)
    dictionary.filter_extremes(no_below=3, no_above=0.7, keep_n=100000)
    corpus = [dictionary.doc2bow(t) for t in tokens]
    return dictionary, corpus

wel_dict, wel_corpus     = build_dict(wel_tokens)
net_dict, net_corpus     = build_dict(net_tokens)
pred_dict, pred_corpus   = build_dict(pred_tokens)


In [8]:
from gensim.models import LdaModel

def train_lda(dictionary, corpus, name):
    lda = LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=100,
        random_state=1000,
        update_every=1,
        chunksize=100,
        passes=3,
        alpha='auto'
    )
    lda.save(f"{folder}/lda_{name}.model")
    return lda


In [11]:
lda_wel  = train_lda(wel_dict, wel_corpus, "welfake")
lda_net  = train_lda(net_dict, net_corpus, "fakenewsnet")
lda_pred = train_lda(pred_dict, pred_corpus, "fakepred")


In [9]:

from gensim.models.coherencemodel import CoherenceModel
import numpy as np

def evaluate_lda(lda, corpus, tokens, dictionary, name):

    coherence_model = CoherenceModel(
        model=lda,
        texts=tokens,
        dictionary=dictionary,
        coherence='c_v'
    )
    coherence = coherence_model.get_coherence()

    perplexity = -lda.log_perplexity(corpus)

    print(f"\n📌 Results for {name}:")
    print(f"Coherence Score: {coherence}")
    print(f"Perplexity:     {perplexity}")

    return coherence, perplexity


In [12]:
evaluate_lda(lda_wel, wel_corpus, wel_tokens, wel_dict, "WELFake")
evaluate_lda(lda_net, net_corpus, net_tokens, net_dict, "FakeNewsNet")
evaluate_lda(lda_pred, pred_corpus, pred_tokens, pred_dict, "FakeNewsPrediction")



📌 Results for WELFake:
Coherence Score: 0.48806110124505275
Perplexity:     27.62189703857191

📌 Results for FakeNewsNet:
Coherence Score: 0.6823746409524571
Perplexity:     100.21639188961674

📌 Results for FakeNewsPrediction:
Coherence Score: 0.4863717123473308
Perplexity:     23.383641230375687


(np.float64(0.4863717123473308), np.float64(23.383641230375687))

In [13]:
def show_topics(lda):
    for idx, topic in lda.print_topics(num_topics=10):
        print(f"\nTopic #{idx}\n{topic}")

show_topics(lda_wel)



Topic #97
0.366*"de" + 0.284*"la" + 0.113*"un" + 0.090*"en" + 0.077*"que" + 0.029*"se" + 0.013*"par" + 0.001*"pdt" + 0.000*"slaughter”" + 0.000*"prattle"

Topic #21
0.166*"russian" + 0.141*"putin" + 0.132*"russia" + 0.070*"election" + 0.054*"ukraine" + 0.053*"vladimir" + 0.036*"campaign" + 0.030*"kremlin" + 0.030*"moscow" + 0.024*"meeting"

Topic #86
0.227*"le" + 0.093*"alex" + 0.074*"et" + 0.063*"phrase" + 0.033*"outright" + 0.032*"pour" + 0.032*"du" + 0.030*"pa" + 0.025*"formation" + 0.024*"powell"

Topic #79
0.395*"russia" + 0.258*"russian" + 0.070*"moscow" + 0.062*"foreign" + 0.046*"russia’s" + 0.028*"soviet" + 0.019*"tie" + 0.018*"ministry" + 0.017*"earnest" + 0.017*"josh"

Topic #99
0.227*"wikileaks" + 0.142*"machine" + 0.118*"leak" + 0.106*"native" + 0.074*"illinois" + 0.072*"lord" + 0.045*"missouri" + 0.029*"revelation" + 0.027*"joseph" + 0.024*"questionable"

Topic #39
0.032*"many" + 0.026*"one" + 0.023*"even" + 0.017*"like" + 0.016*"work" + 0.016*"much" + 0.016*"way" + 0.014